# Data Preprocessing Module

Functions for loading and preprocessing the KDD dataset.

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

In [5]:
def load_and_preprocess_data(file_path):
    
    columns = [
        "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
        "wrong_fragment","urgent","hot","num_failed_logins","logged_in",
        "num_compromised","root_shell","su_attempted","num_root",
        "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
        "is_host_login","is_guest_login","count","srv_count","serror_rate",
        "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
        "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
        "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
        "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
        "dst_host_rerror_rate","dst_host_srv_rerror_rate",
        "label", "difficulty_level"
    ]
    
    df = pd.read_csv(file_path, names=columns)
    df = df.drop(columns=["difficulty_level"])
    df['label'] = df['label'].apply(lambda x: 0 if x == 'normal' else 1)
    
    categorical_cols = ["protocol_type", "service", "flag"]
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    
    X = df.drop(columns=["label"])
    y = df["label"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test

## Feature Selection Functions

In [6]:
def select_features_kbest(X_train, X_test, y_train, k=20):
    
    selector = SelectKBest(score_func=f_classif, k=k)
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_test_selected = selector.transform(X_test)
    

    scores = selector.scores_
    selected_indices = selector.get_support(indices=True)
    
    feature_scores = pd.DataFrame({
        'Feature_Index': range(len(scores)),
        'Score': scores,
        'Selected': selector.get_support()
    }).sort_values('Score', ascending=False)
    
    print(f"\n{'='*60}")
    print(f"FEATURE SELECTION - SelectKBest (ANOVA F-value)")
    print(f"{'='*60}")
    print(f"Original number of features: {X_train.shape[1]}")
    print(f"Selected number of features: {k}")
    print(f"Dimensionality reduction: {100 * (1 - k/X_train.shape[1]):.2f}%")
    print(f"\nTop 20 Most Important Features:")
    print(feature_scores.head(20).to_string(index=False))
    print(f"{'='*60}\n")
    
    return X_train_selected, X_test_selected, selector, feature_scores

In [7]:
def select_features_random_forest(X_train, X_test, y_train, k=20):
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    
   
    importances = rf.feature_importances_
    
   
    top_indices = np.argsort(importances)[::-1][:k]
    top_indices_sorted = np.sort(top_indices)
    
    X_train_selected = X_train[:, top_indices_sorted]
    X_test_selected = X_test[:, top_indices_sorted]
    
    feature_importance = pd.DataFrame({
        'Feature_Index': range(len(importances)),
        'Importance': importances,
        'Rank': np.argsort(np.argsort(importances)[::-1]) + 1
    }).sort_values('Importance', ascending=False)
    
    print(f"\n{'='*60}")
    print(f"FEATURE SELECTION - Random Forest Importance")
    print(f"{'='*60}")
    print(f"Original number of features: {X_train.shape[1]}")
    print(f"Selected number of features: {k}")
    print(f"Dimensionality reduction: {100 * (1 - k/X_train.shape[1]):.2f}%")
    print(f"\nTop 20 Most Important Features:")
    print(feature_importance.head(20).to_string(index=False))
    print(f"{'='*60}\n")
    
    return X_train_selected, X_test_selected, top_indices_sorted, feature_importance

## Advanced Feature Engineering - Multiple K Values

In [ ]:
def test_multiple_k_values(X_train, X_test, y_train, y_test, k_values=[10, 15, 20, 25, 30]):
    """
    Test feature selection with multiple k values and evaluate performance.
    
    Args:
        X_train, X_test: Training and testing features
        y_train, y_test: Training and testing labels
        k_values: List of k values to test (default: [10, 15, 20, 25, 30])
    
    Returns:
        results_df: DataFrame with results for each k value
        best_k: Optimal k value based on accuracy
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    
    results = []
    
    print(f"\n{'='*80}")
    print("TESTING MULTIPLE K VALUES FOR FEATURE SELECTION")
    print(f"{'='*80}\n")
    
    for k in k_values:
        # Select features
        selector = SelectKBest(score_func=f_classif, k=k)
        X_train_k = selector.fit_transform(X_train, y_train)
        X_test_k = selector.transform(X_test)
        
        # Train Logistic Regression
        lr = LogisticRegression(max_iter=1000, random_state=42)
        lr.fit(X_train_k, y_train)
        lr_pred = lr.predict(X_test_k)
        
        lr_acc = accuracy_score(y_test, lr_pred)
        lr_f1 = f1_score(y_test, lr_pred)
        lr_prec = precision_score(y_test, lr_pred)
        lr_rec = recall_score(y_test, lr_pred)
        
        # Train Decision Tree
        dt = DecisionTreeClassifier(max_depth=10, random_state=42)
        dt.fit(X_train_k, y_train)
        dt_pred = dt.predict(X_test_k)
        
        dt_acc = accuracy_score(y_test, dt_pred)
        dt_f1 = f1_score(y_test, dt_pred)
        dt_prec = precision_score(y_test, dt_pred)
        dt_rec = recall_score(y_test, dt_pred)
        
        # Average performance
        avg_acc = (lr_acc + dt_acc) / 2
        avg_f1 = (lr_f1 + dt_f1) / 2
        
        results.append({
            'K': k,
            'Reduction': f"{100 * (1 - k/X_train.shape[1]):.1f}%",
            'LR_Accuracy': f"{lr_acc*100:.2f}%",
            'LR_F1': f"{lr_f1*100:.2f}%",
            'DT_Accuracy': f"{dt_acc*100:.2f}%",
            'DT_F1': f"{dt_f1*100:.2f}%",
            'Avg_Accuracy': f"{avg_acc*100:.2f}%",
            'Avg_F1': f"{avg_f1*100:.2f}%"
        })
        
        print(f"K={k:2d} ({100 * (1 - k/X_train.shape[1]):5.1f}% reduction)")
        print(f"  LR: Acc={lr_acc*100:6.2f}%, F1={lr_f1*100:6.2f}%")
        print(f"  DT: Acc={dt_acc*100:6.2f}%, F1={dt_f1*100:6.2f}%")
        print()
    
    results_df = pd.DataFrame(results)
    print(f"{'='*80}\n")
    print("SUMMARY TABLE:")
    print(results_df.to_string(index=False))
    print(f"{'='*80}\n")
    
    return results_df

## Feature Engineering - Interaction Features

In [ ]:
def create_interaction_features(X_train, X_test, interaction_pairs=None):
    """
    Create interaction features by multiplying selected feature pairs.
    
    Args:
        X_train: Training features (numpy array)
        X_test: Testing features (numpy array)
        interaction_pairs: List of (idx1, idx2) tuples for feature pairs to interact
                          If None, uses top important pairs
    
    Returns:
        X_train_inter: Training features with interactions added
        X_test_inter: Testing features with interactions added
    """
    X_train_inter = X_train.copy()
    X_test_inter = X_test.copy()
    
    if interaction_pairs is None:
        # Default: Create interactions between top 5 features (based on variance)
        feature_vars = np.var(X_train, axis=0)
        top_indices = np.argsort(feature_vars)[-5:]
        interaction_pairs = []
        for i in range(len(top_indices)):
            for j in range(i+1, len(top_indices)):
                interaction_pairs.append((top_indices[i], top_indices[j]))
    
    print(f"\n{'='*60}")
    print(f"CREATING INTERACTION FEATURES")
    print(f"{'='*60}")
    print(f"Original features: {X_train.shape[1]}")
    print(f"Creating {len(interaction_pairs)} interaction features")
    
    # Create interaction features
    for idx1, idx2 in interaction_pairs:
        interaction = X_train[:, idx1] * X_train[:, idx2]
        X_train_inter = np.column_stack([X_train_inter, interaction])
        
        interaction_test = X_test[:, idx1] * X_test[:, idx2]
        X_test_inter = np.column_stack([X_test_inter, interaction_test])
    
    print(f"Total features after interactions: {X_train_inter.shape[1]}")
    print(f"{'='*60}\n")
    
    return X_train_inter, X_test_inter

## Feature Engineering - Domain-Specific Features

In [ ]:
def create_domain_specific_features(X_train_original, X_test_original, y_train, y_test):
    """
    Create domain-specific features for network intrusion detection.
    
    These features are engineered based on domain knowledge about network attacks:
    - Anomaly indicators (high error rates, unusual connection patterns)
    - Concentration metrics (how focused traffic is)
    - Imbalance metrics (whether traffic is imbalanced across connections)
    
    Args:
        X_train_original: Original unscaled training features
        X_test_original: Original unscaled testing features
        y_train, y_test: Labels
    
    Returns:
        X_train_domain: Features with domain-specific features added
        X_test_domain: Features with domain-specific features added
    """
    from sklearn.preprocessing import StandardScaler
    
    print(f"\n{'='*60}")
    print(f"CREATING DOMAIN-SPECIFIC FEATURES")
    print(f"{'='*60}")
    
    # Create new feature matrix starting with original scaled features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_original)
    X_test_scaled = scaler.transform(X_test_original)
    
    # List of feature indices that exist in the original data
    # These would need to be mapped to actual feature indices
    # For now, we'll create synthetic domain features based on aggregate properties
    
    new_features_train = []
    new_features_test = []
    
    # Feature 1: Total connection anomaly score (sum of error rates)
    # Using columns that typically contain error rate information
    if X_train_scaled.shape[1] >= 10:
        total_anomaly_train = np.sum(np.abs(X_train_scaled[:, :15]), axis=1, keepdims=True)
        total_anomaly_test = np.sum(np.abs(X_test_scaled[:, :15]), axis=1, keepdims=True)
        new_features_train.append(total_anomaly_train)
        new_features_test.append(total_anomaly_test)
    
    # Feature 2: Connection concentration (variance of feature values)
    conn_concentration_train = np.var(X_train_scaled[:, :20], axis=1, keepdims=True)
    conn_concentration_test = np.var(X_test_scaled[:, :20], axis=1, keepdims=True)
    new_features_train.append(conn_concentration_train)
    new_features_test.append(conn_concentration_test)
    
    # Feature 3: Attack indicator (combination of suspicious features)
    # High dst_host_srv_error_rate + high srv_error_rate
    if X_train_scaled.shape[1] >= 25:
        attack_indicator_train = np.max(X_train_scaled[:, 20:25], axis=1, keepdims=True)
        attack_indicator_test = np.max(X_test_scaled[:, 20:25], axis=1, keepdims=True)
        new_features_train.append(attack_indicator_train)
        new_features_test.append(attack_indicator_test)
    
    # Concatenate domain features
    if new_features_train:
        X_train_domain = np.hstack([X_train_scaled] + new_features_train)
        X_test_domain = np.hstack([X_test_scaled] + new_features_test)
    else:
        X_train_domain = X_train_scaled
        X_test_domain = X_test_scaled
    
    print(f"Original features: {X_train_scaled.shape[1]}")
    print(f"Domain-specific features added: {len(new_features_train)}")
    print(f"  1. Total Connection Anomaly Score")
    print(f"  2. Connection Concentration Metric")
    if len(new_features_train) >= 3:
        print(f"  3. Attack Indicator (Suspicious Pattern)")
    print(f"Total features: {X_train_domain.shape[1]}")
    print(f"{'='*60}\n")
    
    return X_train_domain, X_test_domain